# Notebook 1: LightGlue Tam Değerlendirme + Baseline Karşılaştırma + Ablation Study

Bu sürüm v5 ana model artifact'leriyle uyumludur:
- `dinov2_vitb14`
- CLS token pooling
- `/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final`
- `dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth`

LightGlue burada ana karar mekanizması değil, DINOv2 Top-K adaylarının geometrik re-ranking ile nasıl değiştiğini ölçen ek tanı testidir.

In [ ]:
# ============================================================
# Hücre 1: Kütüphane Kurulumu
# ============================================================
!pip install faiss-cpu folium pytorch-metric-learning scikit-learn -q
!pip install git+https://github.com/cvg/LightGlue.git -q
print("✅ Tüm kütüphaneler kuruldu.")


In [ ]:
# ============================================================
# Hücre 2: Import'lar ve Konfigürasyon
# ============================================================
import os, json, random, zipfile, shutil, math, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from PIL import Image, ImageOps
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from pytorch_metric_learning import losses, miners
from pytorch_metric_learning.samplers import MPerClassSampler
from sklearn.preprocessing import LabelEncoder

import faiss

# LightGlue
from lightglue import LightGlue, ALIKED
from lightglue.utils import load_image, rbd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ─── Konfigürasyon ──────────────────────────────────────────
DRIVE_ZIP     = "/content/drive/My Drive/kirsehir_data.zip"
LOCAL_ROOT    = "/content/kirsehir_data"
OUTPUT_DIR    = "/content/drive/My Drive/Kirsehir_VPR_DINOv2_v5_Final"
RESULTS_DIR   = "/content/drive/My Drive/Kirsehir_VPR_Results"

MODEL_NAME    = "dinov2_vitb14"
MODEL_POOLING = "cls"
MODEL_CHECKPOINT_NAME = "dinov2_kirsehir_v5_vitb14_cls_domainaug_best.pth"
DRIVE_MODEL_PATH = os.path.join(OUTPUT_DIR, MODEL_CHECKPOINT_NAME)

OUTFEATURES   = 512
IMG_SIZE      = 518
BATCH_SIZE    = 128
NUM_WORKERS   = 4
UNFREEZE_LAST_N_BLOCKS = 4

TOP_K         = 20
RERANK_TOP_K  = 20
GRID_SIZE     = 0.002

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Cihaz: {DEVICE}")
print(f"✅ v5 model: {MODEL_NAME} / {MODEL_POOLING}")

In [ ]:
# ============================================================
# Hücre 3: Veri Yükleme ve Bölme (Orijinal Split - Aynı SEED)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(LOCAL_ROOT):
    LOCAL_ZIP = "/content/kirsehir_data.zip"
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zf:
        zf.extractall(LOCAL_ROOT)
    os.remove(LOCAL_ZIP)

entries = os.listdir(LOCAL_ROOT)
if len(entries) == 1 and os.path.isdir(os.path.join(LOCAL_ROOT, entries[0])):
    LOCAL_ROOT = os.path.join(LOCAL_ROOT, entries[0])

def extract_lat_lon_heading(filepath):
    parts = os.path.splitext(os.path.basename(filepath))[0].split("_")
    try:
        return float(parts[0]), float(parts[1]), float(parts[2].replace("h", "") if len(parts)>2 else "0")
    except: return None, None, None

all_images = []
for street_name in sorted(os.listdir(LOCAL_ROOT)):
    street_path = os.path.join(LOCAL_ROOT, street_name)
    if not os.path.isdir(street_path) or street_name == "model": continue
    for fname in sorted(os.listdir(street_path)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")): continue
        fpath = os.path.join(street_path, fname)
        lat, lon, heading = extract_lat_lon_heading(fpath)
        if lat is not None:
            all_images.append({
                "filepath": fpath, "street": street_name,
                "lat": lat, "lon": lon, "heading": heading,
                "point_id": f"{lat:.6f}_{lon:.6f}",
                "block_id": f"{int(lat/GRID_SIZE)}_{int(lon/GRID_SIZE)}"
            })

df_all = pd.DataFrame(all_images).sort_values(by=["point_id", "heading"])

unique_blocks = df_all["block_id"].unique().tolist()
random.shuffle(unique_blocks)
n_blocks = len(unique_blocks)

train_blocks = set(unique_blocks[:int(n_blocks * 0.70)])
test_blocks  = set(unique_blocks[int(n_blocks * 0.70):])

# Train seti
df_train = df_all[df_all["block_id"].isin(train_blocks)].copy()
df_train["class_id"] = df_train["point_id"]
le = LabelEncoder()
df_train["label"] = le.fit_transform(df_train["class_id"])

# Test seti: Query/DB bölünmesi
df_test = df_all[df_all["block_id"].isin(test_blocks)].copy()
point_counts = df_test['point_id'].value_counts()
valid_points = point_counts[point_counts >= 2].index
df_test = df_test[df_test['point_id'].isin(valid_points)]

df_query = df_test.groupby("point_id").tail(1).reset_index(drop=True)
query_filepaths = set(df_query["filepath"])
df_db = df_test[~df_test["filepath"].isin(query_filepaths)].reset_index(drop=True)

print(f"\n✅ Toplam Veri: {len(df_all):,}")
print(f"🚀 Eğitim (Train): {len(df_train)}")
print(f"🗺️ Harita (DB)   : {len(df_db)}")
print(f"🔍 Sorgu (Query) : {len(df_query)}")


In [ ]:
# ============================================================
# Hücre 4: Model Tanımları ve Yardımcı Fonksiyonlar
# ============================================================

def open_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=1).pow(1.0 / self.p)

class VPRDINOv2(nn.Module):
    """DINOv2 + CLS/GeM pooling + v5 projection head."""
    def __init__(self, backbone_name, out_dim, pooling="cls", freeze_all=False,
                 unfreeze_last_n=4, head_dropout=0.10):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, verbose=False)
        self.pooling_type = pooling.lower()

        for p in self.backbone.parameters():
            p.requires_grad = False

        if not freeze_all:
            n_blocks = len(getattr(self.backbone, "blocks", []))
            trainable_blocks = set(range(max(0, n_blocks - unfreeze_last_n), n_blocks))
            for name, param in self.backbone.named_parameters():
                if name.startswith("norm"):
                    param.requires_grad = True
                elif name.startswith("blocks."):
                    parts = name.split(".")
                    if len(parts) > 1 and parts[1].isdigit() and int(parts[1]) in trainable_blocks:
                        param.requires_grad = True

        if self.pooling_type == "gem":
            self.pool = GeMPooling(p=3)
        elif self.pooling_type != "cls":
            raise ValueError(f"Desteklenmeyen pooling: {pooling}")

        embed_dim = self.backbone.embed_dim
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=head_dropout),
            nn.Linear(embed_dim, out_dim),
        )

    def forward(self, x):
        ret = self.backbone.forward_features(x)
        if self.pooling_type == "gem":
            x = self.pool(ret["x_norm_patchtokens"])
        else:
            x = ret["x_norm_clstoken"]
        x = self.head(x)
        return nn.functional.normalize(x, p=2, dim=1)


class StandardVPRDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        return self.transform(open_rgb_image(self.df.iloc[idx]["filepath"])), idx

class VPRTrainDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(open_rgb_image(row["filepath"])), int(row["label"])


eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.55, 1.0), ratio=(0.75, 1.33),
                        interpolation=T.InterpolationMode.BICUBIC),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.18, p=1.0,
                                        interpolation=T.InterpolationMode.BICUBIC)], p=0.35),
    T.RandomRotation(degrees=7, interpolation=T.InterpolationMode.BICUBIC),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.30, hue=0.04),
    T.RandomAutocontrast(p=0.25),
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.20),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2, dp, dl = map(math.radians, [lat1, lat2, lat2-lat1, lon2-lon1])
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


@torch.no_grad()
def extract_embeddings(model, df, transform, bs, device, desc="Emb"):
    loader = DataLoader(StandardVPRDataset(df, transform), batch_size=bs,
                        num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    embs = []
    for imgs, _ in tqdm(loader, desc=desc):
        imgs = imgs.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            embs.append(model(imgs).detach().cpu().numpy())
    return np.concatenate(embs, axis=0)


def evaluate_retrieval(query_embeddings, db_embeddings, df_query, df_db, top_k=20, method_name="Model"):
    """Standart VPR değerlendirme fonksiyonu. Ana metrik Top-1'dir."""
    db_emb = db_embeddings.copy().astype(np.float32)
    qry_emb = query_embeddings.copy().astype(np.float32)
    faiss.normalize_L2(db_emb)
    faiss.normalize_L2(qry_emb)

    index = faiss.IndexFlatIP(db_emb.shape[1])
    index.add(db_emb)
    dists, idxs = index.search(qry_emb, top_k)

    db_meta = df_db[["filepath","street","lat","lon","heading","point_id"]].to_dict("records")

    errors_top1 = []
    errors_spatial = []
    correct_street = 0
    street_topk = {5: 0, 10: 0, 20: 0}

    for i in range(len(df_query)):
        q_row = df_query.iloc[i]
        q_lat, q_lon = q_row["lat"], q_row["lon"]

        best = db_meta[int(idxs[i, 0])]
        err = haversine(q_lat, q_lon, best["lat"], best["lon"])
        errors_top1.append(err)

        if best["street"] == q_row["street"]:
            correct_street += 1

        top_streets = [db_meta[int(j)]["street"] for j in idxs[i]]
        for k in street_topk:
            if q_row["street"] in top_streets[:min(k, len(top_streets))]:
                street_topk[k] += 1

        k_spatial = min(5, top_k)
        top_matches = [db_meta[int(idxs[i, k])] for k in range(k_spatial)]
        w = np.array([dists[i, k] for k in range(k_spatial)])
        w = np.maximum(w, 0); w /= (w.sum() + 1e-9)
        pred_lat = sum(m["lat"] * wi for m, wi in zip(top_matches, w))
        pred_lon = sum(m["lon"] * wi for m, wi in zip(top_matches, w))
        errors_spatial.append(haversine(q_lat, q_lon, pred_lat, pred_lon))

    errors_top1 = np.array(errors_top1)
    errors_spatial = np.array(errors_spatial)
    n = max(1, len(df_query))

    return {
        "method": method_name,
        "street_accuracy": correct_street / n * 100,
        "street_top5_accuracy": street_topk[5] / n * 100,
        "street_top10_accuracy": street_topk[10] / n * 100,
        "street_top20_accuracy": street_topk[20] / n * 100,
        "median_error_top1": np.median(errors_top1),
        "mean_error_top1": np.mean(errors_top1),
        "median_error_spatial": np.median(errors_spatial),
        "mean_error_spatial": np.mean(errors_spatial),
        "recall_25m": (errors_top1 <= 25).mean() * 100,
        "recall_50m": (errors_top1 <= 50).mean() * 100,
        "recall_100m": (errors_top1 <= 100).mean() * 100,
        "recall_500m": (errors_top1 <= 500).mean() * 100,
        "errors_top1": errors_top1,
        "errors_spatial": errors_spatial,
        "dists": dists,
        "idxs": idxs,
        "db_meta": db_meta
    }

print("✅ v5 model tanımları ve yardımcı fonksiyonlar hazır.")

In [ ]:
# ============================================================
# Hücre 5: v5 Eğitilmiş Modeli Yükle + Embedding Çıkar
# ============================================================
if not os.path.exists(DRIVE_MODEL_PATH):
    raise FileNotFoundError(
        f"v5 checkpoint bulunamadı:\n{DRIVE_MODEL_PATH}\n"
        "Önce v5 ana eğitim notebook'unu çalıştır."
    )

model_finetuned = VPRDINOv2(
    MODEL_NAME,
    OUTFEATURES,
    pooling=MODEL_POOLING,
    freeze_all=False,
    unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS,
).to(DEVICE)

state_dict = torch.load(DRIVE_MODEL_PATH, map_location=DEVICE)
if isinstance(state_dict, dict) and "model_state" in state_dict:
    state_dict = state_dict["model_state"]
model_finetuned.load_state_dict(state_dict, strict=True)
model_finetuned.eval()
print("✅ v5 fine-tuned model yüklendi.")

torch.cuda.empty_cache()

t0 = time.time()
db_embeddings_ft = extract_embeddings(model_finetuned, df_db, eval_transform, BATCH_SIZE, DEVICE, "DB (v5 Fine-tuned)")
t_db = time.time() - t0

t0 = time.time()
query_embeddings_ft = extract_embeddings(model_finetuned, df_query, eval_transform, BATCH_SIZE, DEVICE, "Query (v5 Fine-tuned)")
t_query = time.time() - t0

print(f"\n⏱️ DB embedding süresi: {t_db:.1f}s ({len(df_db)} görüntü)")
print(f"⏱️ Query embedding süresi: {t_query:.1f}s ({len(df_query)} görüntü)")
print(f"⏱️ Tek görüntü embedding süresi: {t_query/len(df_query)*1000:.1f}ms")

results_ft = evaluate_retrieval(
    query_embeddings_ft,
    db_embeddings_ft,
    df_query,
    df_db,
    top_k=TOP_K,
    method_name="DINOv2 v5 Fine-tuned (ViT-B + CLS)"
)

print(f"\n{'='*55}")
print(f"📊 DINOv2 v5 Fine-tuned Sonuçları")
print(f"{'='*55}")
print(f"   Sokak Doğruluğu  : {results_ft['street_accuracy']:.2f}%")
print(f"   Sokak Top-5/10/20: {results_ft['street_top5_accuracy']:.2f}% / {results_ft['street_top10_accuracy']:.2f}% / {results_ft['street_top20_accuracy']:.2f}%")
print(f"   Medyan Hata (R@1): {results_ft['median_error_top1']:.1f}m")
print(f"   R@1 <25m : {results_ft['recall_25m']:.2f}%")
print(f"   R@1 <100m: {results_ft['recall_100m']:.2f}%")

In [ ]:
# ============================================================
# Hücre 6: LightGlue Re-Ranking — TÜM TEST SETİ
# ============================================================
# DİKKAT: Bu hücre uzun sürebilir (query sayısı x 20 aday x LightGlue)
# Tahmini süre: ~1-3 saat (query sayısına bağlı)

lg_extractor = ALIKED(max_num_keypoints=1024).eval().to(DEVICE)
lg_matcher = LightGlue(features='aliked').eval().to(DEVICE)

# FAISS araması — fine-tuned model ile Top-20 adayları al
db_emb_n = db_embeddings_ft.copy().astype(np.float32)
qry_emb_n = query_embeddings_ft.copy().astype(np.float32)
faiss.normalize_L2(db_emb_n)
faiss.normalize_L2(qry_emb_n)

faiss_index = faiss.IndexFlatIP(OUTFEATURES)
faiss_index.add(db_emb_n)
dists_all, idxs_all = faiss_index.search(qry_emb_n, RERANK_TOP_K)

db_meta = df_db[["filepath","street","lat","lon","heading","point_id"]].to_dict("records")

# ─── LightGlue ile tüm sorguları yeniden sırala ───
print(f"\n{'='*60}")
print(f"⚡ LightGlue Re-Ranking başlıyor ({len(df_query)} sorgu x {RERANK_TOP_K} aday)")
print(f"{'='*60}")

lg_results_all = []  # Her sorgu için re-ranking sonuçları
lg_timings = []      # Her sorgu için süre

for qi in tqdm(range(len(df_query)), desc="LightGlue Re-Ranking"):
    q_row = df_query.iloc[qi]
    t_start = time.time()

    try:
        img0 = load_image(q_row["filepath"]).to(DEVICE)
        feats0 = lg_extractor.extract(img0)
    except Exception as e:
        print(f"⚠️ Sorgu {qi} yüklenemedi: {e}")
        lg_results_all.append(None)
        lg_timings.append(0)
        continue

    candidates = []
    for k in range(RERANK_TOP_K):
        cand_meta = db_meta[idxs_all[qi, k]]
        try:
            img1 = load_image(cand_meta["filepath"]).to(DEVICE)
            feats1 = lg_extractor.extract(img1)

            matches01 = lg_matcher({"image0": feats0, "image1": feats1})
            matches_clean = rbd(matches01)
            num_matches = len(matches_clean['matches'])

            # Geometrik doğrulama: inlier ratio hesapla
            match_confidence = matches_clean.get('matching_scores0', None)
            if match_confidence is not None and len(match_confidence) > 0:
                avg_confidence = float(match_confidence[matches_clean['matches'][..., 0]].mean()) if num_matches > 0 else 0.0
            else:
                avg_confidence = 0.0

            candidates.append({
                "original_rank": k + 1,
                "filepath": cand_meta["filepath"],
                "street": cand_meta["street"],
                "lat": cand_meta["lat"],
                "lon": cand_meta["lon"],
                "num_matches": num_matches,
                "avg_confidence": avg_confidence,
                "dino_score": float(dists_all[qi, k]),
                # Hibrit skor: normalize edilmiş match sayısı * ortalama güven
                "hybrid_score": num_matches * (1.0 + avg_confidence)
            })

            del img1, feats1, matches01
        except Exception as e:
            candidates.append({
                "original_rank": k + 1,
                "filepath": cand_meta["filepath"],
                "street": cand_meta["street"],
                "lat": cand_meta["lat"],
                "lon": cand_meta["lon"],
                "num_matches": 0,
                "avg_confidence": 0.0,
                "dino_score": float(dists_all[qi, k]),
                "hybrid_score": 0.0
            })

    del img0, feats0
    torch.cuda.empty_cache()

    # Üç farklı sıralama kriteri ile sonuçları sakla
    lg_results_all.append(candidates)
    lg_timings.append(time.time() - t_start)

    # Her 50 sorguda ara rapor
    if (qi + 1) % 50 == 0:
        avg_time = np.mean(lg_timings[-50:])
        remaining = (len(df_query) - qi - 1) * avg_time / 60
        print(f"   [{qi+1}/{len(df_query)}] Ort. süre: {avg_time:.1f}s/sorgu, Kalan: ~{remaining:.0f} dk")

print(f"\n✅ LightGlue tamamlandı. Toplam süre: {sum(lg_timings)/60:.1f} dakika")
print(f"   Ortalama süre/sorgu: {np.mean(lg_timings):.1f}s")


In [ ]:
# ============================================================
# Hücre 7: LightGlue Sonuçlarını Değerlendir (3 Kriter Karşılaştırması)
# ============================================================

def evaluate_reranking(lg_results_all, df_query, sort_key, method_name):
    """Re-ranking sonuçlarını verilen sıralama kriterine göre değerlendir."""
    errors_top1 = []
    correct_street = 0
    valid_count = 0

    for qi in range(len(df_query)):
        if lg_results_all[qi] is None:
            continue
        valid_count += 1
        q_row = df_query.iloc[qi]

        # Verilen kritere göre sırala
        sorted_cands = sorted(lg_results_all[qi], key=lambda x: x[sort_key], reverse=True)
        best = sorted_cands[0]

        err = haversine(q_row["lat"], q_row["lon"], best["lat"], best["lon"])
        errors_top1.append(err)

        if best["street"] == q_row["street"]:
            correct_street += 1

    errors_top1 = np.array(errors_top1)

    return {
        "method": method_name,
        "street_accuracy": correct_street / valid_count * 100,
        "median_error_top1": np.median(errors_top1),
        "mean_error_top1": np.mean(errors_top1),
        "recall_25m": (errors_top1 <= 25).mean() * 100,
        "recall_50m": (errors_top1 <= 50).mean() * 100,
        "recall_100m": (errors_top1 <= 100).mean() * 100,
        "recall_500m": (errors_top1 <= 500).mean() * 100,
        "errors_top1": errors_top1,
        "valid_count": valid_count
    }

# Üç farklı re-ranking kriteri
res_lg_matches = evaluate_reranking(lg_results_all, df_query, "num_matches",
                                     "DINOv2 + LightGlue (Match Count)")
res_lg_hybrid = evaluate_reranking(lg_results_all, df_query, "hybrid_score",
                                    "DINOv2 + LightGlue (Hybrid Score)")

# DINOv2-only (re-ranking yok, orijinal DINO sıralaması)
res_lg_dino_only = evaluate_reranking(lg_results_all, df_query, "dino_score",
                                       "DINOv2 Only (Top-20 içinden)")

print(f"\n{'='*70}")
print(f"📊 LightGlue RE-RANKING KARŞILAŞTIRMASI (Tüm Test Seti)")
print(f"{'='*70}")
print(f"{'Yöntem':<45} {'Sokak%':>7} {'Med.Hata':>9} {'R@1<25m':>8} {'R@1<100m':>9}")
print(f"{'-'*70}")
for r in [res_lg_dino_only, res_lg_matches, res_lg_hybrid]:
    print(f"{r['method']:<45} {r['street_accuracy']:>6.2f}% {r['median_error_top1']:>8.1f}m "
          f"{r['recall_25m']:>7.2f}% {r['recall_100m']:>8.2f}%")

# LightGlue zamanlamaları
print(f"\n⏱️ LightGlue Re-Ranking Zamanlama:")
print(f"   Ortalama süre/sorgu (20 aday): {np.mean(lg_timings):.2f}s")
print(f"   Medyan süre/sorgu: {np.median(lg_timings):.2f}s")


In [ ]:
# ============================================================
# Hücre 8: BASELINE 1 — Frozen DINOv2 v5 Backbone
# ============================================================
print("⏳ Frozen DINOv2 ViT-B/CLS baseline yükleniyor...")

model_frozen = VPRDINOv2(
    MODEL_NAME, OUTFEATURES, pooling=MODEL_POOLING,
    freeze_all=True, unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS
).to(DEVICE)

# Bu baseline'da backbone tamamen frozen, sadece projection head eğitilir.
model_frozen.train()
train_dataset = VPRTrainDataset(df_train, train_transform)
sampler = MPerClassSampler(df_train["label"].values, m=4, batch_size=BATCH_SIZE,
                           length_before_new_iter=len(df_train))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

miner_fn = miners.MultiSimilarityMiner(epsilon=0.1)
loss_fn = losses.MultiSimilarityLoss(alpha=2, beta=50, base=0.5)
optimizer = optim.AdamW(model_frozen.head.parameters(), lr=8e-5, weight_decay=1e-4)

EPOCHS_BASELINE = 5
print(f"\n⏳ Sadece head eğitimi başlıyor ({EPOCHS_BASELINE} epoch)...")
for epoch in range(EPOCHS_BASELINE):
    model_frozen.train()
    epoch_loss, valid_batches = 0.0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Frozen Baseline Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model_frozen(imgs)
        hard_pairs = miner_fn(embeddings, labels)
        loss = loss_fn(embeddings, labels, hard_pairs)
        if torch.isfinite(loss) and loss.item() > 0:
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); valid_batches += 1
    avg = epoch_loss / valid_batches if valid_batches > 0 else 0
    print(f"   Epoch {epoch+1} Loss: {avg:.4f}")

model_frozen.eval()
torch.cuda.empty_cache()

db_emb_frozen = extract_embeddings(model_frozen, df_db, eval_transform, BATCH_SIZE, DEVICE, "DB (Frozen v5)")
qry_emb_frozen = extract_embeddings(model_frozen, df_query, eval_transform, BATCH_SIZE, DEVICE, "Query (Frozen v5)")

results_frozen = evaluate_retrieval(
    qry_emb_frozen, db_emb_frozen, df_query, df_db,
    top_k=TOP_K, method_name="DINOv2 ViT-B Frozen Backbone (CLS Head)"
)

print(f"\n📊 Frozen DINOv2 Sonuçları:")
print(f"   Sokak Doğruluğu  : {results_frozen['street_accuracy']:.2f}%")
print(f"   Medyan Hata (R@1): {results_frozen['median_error_top1']:.1f}m")
print(f"   R@1 <100m: {results_frozen['recall_100m']:.2f}%")

del model_frozen; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ============================================================
# Hücre 9: ABLATION 1 — Kısa CLS Fine-Tuning
# ============================================================
print("⏳ ViT-B + CLS kısa fine-tuning ablation eğitiliyor...")

model_cls = VPRDINOv2(
    MODEL_NAME, OUTFEATURES, pooling="cls",
    freeze_all=False, unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS
).to(DEVICE)

train_dataset = VPRTrainDataset(df_train, train_transform)
sampler = MPerClassSampler(df_train["label"].values, m=4, batch_size=BATCH_SIZE,
                           length_before_new_iter=len(df_train))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

miner_fn = miners.MultiSimilarityMiner(epsilon=0.1)
loss_fn = losses.MultiSimilarityLoss(alpha=2, beta=50, base=0.5)
optimizer = optim.AdamW([
    {'params': [p for p in model_cls.backbone.parameters() if p.requires_grad], 'lr': 5e-6},
    {'params': model_cls.head.parameters(), 'lr': 8e-5}
], weight_decay=1e-4)

EPOCHS_ABL = 5
for epoch in range(EPOCHS_ABL):
    model_cls.train()
    epoch_loss, valid_batches = 0.0, 0
    for imgs, labels in tqdm(train_loader, desc=f"CLS Ablation Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model_cls(imgs)
        hard_pairs = miner_fn(embeddings, labels)
        loss = loss_fn(embeddings, labels, hard_pairs)
        if torch.isfinite(loss) and loss.item() > 0:
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); valid_batches += 1
    avg = epoch_loss / valid_batches if valid_batches > 0 else 0
    print(f"   Epoch {epoch+1} Loss: {avg:.4f}")

model_cls.eval(); torch.cuda.empty_cache()

db_emb_cls = extract_embeddings(model_cls, df_db, eval_transform, BATCH_SIZE, DEVICE, "DB (CLS ablation)")
qry_emb_cls = extract_embeddings(model_cls, df_query, eval_transform, BATCH_SIZE, DEVICE, "Query (CLS ablation)")

results_cls = evaluate_retrieval(
    qry_emb_cls, db_emb_cls, df_query, df_db,
    top_k=TOP_K, method_name="DINOv2 ViT-B + CLS (5 epoch ablation)"
)

print(f"\n📊 CLS Ablation Sonuçları:")
print(f"   Sokak Doğruluğu  : {results_cls['street_accuracy']:.2f}%")
print(f"   Medyan Hata (R@1): {results_cls['median_error_top1']:.1f}m")
print(f"   R@1 <100m: {results_cls['recall_100m']:.2f}%")

del model_cls; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ============================================================
# Hücre 10: ABLATION 2 — Triplet Loss vs Multi-Similarity Loss
# ============================================================
print("⏳ ViT-B + CLS + Triplet Loss modeli eğitiliyor...")

model_triplet = VPRDINOv2(
    MODEL_NAME, OUTFEATURES, pooling=MODEL_POOLING,
    freeze_all=False, unfreeze_last_n=UNFREEZE_LAST_N_BLOCKS
).to(DEVICE)

train_dataset = VPRTrainDataset(df_train, train_transform)
sampler = MPerClassSampler(df_train["label"].values, m=4, batch_size=BATCH_SIZE,
                           length_before_new_iter=len(df_train))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

from pytorch_metric_learning.miners import TripletMarginMiner
miner_triplet = TripletMarginMiner(margin=0.2, type_of_triplets="hard")
loss_triplet = losses.TripletMarginLoss(margin=0.2)

optimizer = optim.AdamW([
    {'params': [p for p in model_triplet.backbone.parameters() if p.requires_grad], 'lr': 5e-6},
    {'params': model_triplet.head.parameters(), 'lr': 8e-5}
], weight_decay=1e-4)

EPOCHS_ABL = 5
for epoch in range(EPOCHS_ABL):
    model_triplet.train()
    epoch_loss, valid_batches = 0.0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Triplet Ablation Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        embeddings = model_triplet(imgs)
        hard_pairs = miner_triplet(embeddings, labels)
        loss = loss_triplet(embeddings, labels, hard_pairs)
        if torch.isfinite(loss) and loss.item() > 0:
            loss.backward(); optimizer.step()
            epoch_loss += loss.item(); valid_batches += 1
    avg = epoch_loss / valid_batches if valid_batches > 0 else 0
    print(f"   Epoch {epoch+1} Loss: {avg:.4f}")

model_triplet.eval(); torch.cuda.empty_cache()

db_emb_trip = extract_embeddings(model_triplet, df_db, eval_transform, BATCH_SIZE, DEVICE, "DB (Triplet)")
qry_emb_trip = extract_embeddings(model_triplet, df_query, eval_transform, BATCH_SIZE, DEVICE, "Query (Triplet)")

results_triplet = evaluate_retrieval(
    qry_emb_trip, db_emb_trip, df_query, df_db,
    top_k=TOP_K, method_name="DINOv2 ViT-B + CLS + Triplet Loss"
)

print(f"\n📊 Triplet Loss Ablation Sonuçları:")
print(f"   Sokak Doğruluğu  : {results_triplet['street_accuracy']:.2f}%")
print(f"   Medyan Hata (R@1): {results_triplet['median_error_top1']:.1f}m")
print(f"   R@1 <100m: {results_triplet['recall_100m']:.2f}%")

del model_triplet; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ============================================================
# Hücre 11: TÜM SONUÇLARIN BİRLEŞİK TABLOSU
# ============================================================

all_results = [
    results_frozen,
    results_cls,
    results_triplet,
    results_ft,
    res_lg_matches,
    res_lg_hybrid,
]

print(f"\n{'='*100}")
print(f"📊 v5 MAKALE İÇİN KARŞILAŞTIRMA TABLOSU")
print(f"{'='*100}")
print(f"{'Yöntem':<50} {'Sokak%':>7} {'Top20%':>7} {'Med.(m)':>8} {'<25m':>7} {'<50m':>7} {'<100m':>7} {'<500m':>7}")
print(f"{'-'*100}")

for r in all_results:
    top20 = r.get("street_top20_accuracy", np.nan)
    top20_str = f"{top20:>6.2f}%" if not np.isnan(top20) else "   n/a "
    print(f"{r['method']:<50} {r['street_accuracy']:>6.2f}% {top20_str} {r['median_error_top1']:>7.1f} "
          f"{r['recall_25m']:>6.2f}% {r['recall_50m']:>6.2f}% {r['recall_100m']:>6.2f}% {r['recall_500m']:>6.2f}%")

print(f"\n{'='*70}")
print(f"📊 SPATIAL VOTING ETKİSİ (v5 Fine-tuned Model)")
print(f"{'='*70}")
print(f"   Top-1 Medyan Hata    : {results_ft['median_error_top1']:.1f}m")
print(f"   Spatial (Top-5) Medyan: {results_ft['median_error_spatial']:.1f}m")
print(f"   Fark                 : {results_ft['median_error_top1'] - results_ft['median_error_spatial']:.1f}m")

results_df = pd.DataFrame([{
    "Method": r["method"],
    "Street Accuracy (%)": round(r["street_accuracy"], 2),
    "Street Top-20 Accuracy (%)": round(r.get("street_top20_accuracy", np.nan), 2),
    "Median Error (m)": round(r["median_error_top1"], 1),
    "Recall@1 <25m (%)": round(r["recall_25m"], 2),
    "Recall@1 <50m (%)": round(r["recall_50m"], 2),
    "Recall@1 <100m (%)": round(r["recall_100m"], 2),
    "Recall@1 <500m (%)": round(r["recall_500m"], 2),
} for r in all_results])

csv_path = os.path.join(RESULTS_DIR, "v5_comparison_table.csv")
results_df.to_csv(csv_path, index=False)
print(f"\n✅ Sonuç tablosu kaydedildi: {csv_path}")

In [ ]:
# ============================================================
# Hücre 12: CDF Karşılaştırma Grafiği (Makale Figürü)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_data = [
    (results_frozen, "Frozen ViT-B CLS", "gray", "--"),
    (results_cls, "Short CLS Ablation", "orange", "-."),
    (results_triplet, "Triplet Loss", "brown", ":"),
    (results_ft, "v5 ViT-B CLS (Ours)", "blue", "-"),
    (res_lg_hybrid, "+ LightGlue Hybrid", "red", "-"),
]

for r, label, color, ls in plot_data:
    sd = np.sort(r["errors_top1"])
    cdf = np.arange(1, len(sd)+1) / len(sd) * 100
    axes[0].plot(sd, cdf, lw=2, color=color, linestyle=ls, label=label)

axes[0].set_xlim(0, 500)
axes[0].set_xlabel("Localization Error (meters)", fontsize=12)
axes[0].set_ylabel("Percentage of Queries (%)", fontsize=12)
axes[0].set_title("Cumulative Distribution Function — v5 Methods", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=100, color='gray', linestyle=':', alpha=0.5)

sd1 = np.sort(results_ft["errors_top1"])
cdf1 = np.arange(1, len(sd1)+1) / len(sd1) * 100
sd2 = np.sort(results_ft["errors_spatial"])
cdf2 = np.arange(1, len(sd2)+1) / len(sd2) * 100

axes[1].plot(sd1, cdf1, lw=2, color="blue", label="Top-1 Only")
axes[1].plot(sd2, cdf2, lw=2, color="green", label="Spatial Voting (Top-5)")
axes[1].set_xlim(0, 500)
axes[1].set_xlabel("Localization Error (meters)", fontsize=12)
axes[1].set_ylabel("Percentage of Queries (%)", fontsize=12)
axes[1].set_title("Effect of Spatial Voting on v5", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, "v5_cdf_comparison.png")
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Figür kaydedildi: {fig_path}")